In [ ]:
import os
import pickle
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests
from pyalex import Works

%load_ext autoreload
%autoreload 2
input_query_openalex = """
"Fuzzy logic based approach for ship-bridge collision alert system"
"""
out_file_openalex = Path("..") / "tests" / "data" / "01_test_search_openalex.pickle"
out_file_openalex.parent.mkdir(parents=True, exist_ok=True)
work_openalex = Works().search_filter(title_and_abstract=input_query_openalex)
with open(out_file_openalex, "wb") as f:
    pickle.dump(work_openalex, f)

In [ ]:
from slrrag.querier_openalex import work_openalex_to_list

l_test_openalex = work_openalex_to_list(work_openalex)
out_file_openalex_list = (
    Path("..") / "tests" / "data" / "01_test_search_openalex_list.pickle"
)
print(out_file_openalex_list)
with open(out_file_openalex_list, "wb") as f:
    pickle.dump(l_test_openalex, f)
l_test_openalex

In [ ]:
from slrrag.querier_openalex import query_openalex

re = query_openalex(input_query_openalex)
re

query

In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
input_query_test_scopus = """
"Fuzzy logic based approach for ship-bridge collision alert system"
"""
input_query_test_scopus = f"""
TITLE-ABS-KEY( {input_query_test_scopus})
"""
scopus_search = ScopusSearch(
    query=input_query_test_scopus, verbose=True, subscriber=True, sort="+relevancy"
)
scopus_search.get_results_size()

In [ ]:
scopus_search.results[0].title.__contains__("R")

In [ ]:
[
    x
    for x in scopus_search.results
    if x.title.__contains__("Risk Assessment of Collision Between Out of")
][0].subtypeDescription

In [ ]:
from slrrag.querier_scopus import scopus_to_dict_list

df_scopus = scopus_to_dict_list(scopus_search)
df_scopus

In [ ]:
from pybliometrics.scopus import ScopusSearch, AbstractRetrieval


# For a single document
def complete_scopus_infos(id_paper: str):
    doc = AbstractRetrieval(identifier=id_paper, view="FULL")
    area_text = " | ".join([x.area for x in doc.subject_areas])
    return pd.Series((doc.language, area_text))


df_scopus[["language", "subject_area"]] = df_scopus["id_scopus"].apply(
    complete_scopus_infos
)
df_scopus

In [ ]:
df_scopus[["language", "subject_area"]]

In [ ]:
df_scopus["id_scopus"].apply(complete_scopus_infos)

In [ ]:
import pickle
from pathlib import Path

out_file_scopus = Path("..") / "tests" / "data" / "01_test_search_scopus.pickle"
with open(out_file_scopus, "wb") as f:
    pickle.dump(scopus_search, f)

In [ ]:
with open(out_file_scopus, "rb") as f:
    t = pickle.load(f)
t.results

In [ ]:
t

In [ ]:
import pybliometrics
from pybliometrics.scopus import ScopusSearch

pybliometrics.init()  # Init for scopus
scopus_search = ScopusSearch(
    query='title("Navigational Risk assessment")',
    verbose=True,
    subscriber=False,
    sort="-relevance",
    refresh=True,
)
scopus_search

In [ ]:
data = [
    {
        "doi": paper.doi,
        "title": paper.title,
        "date": paper.coverDate,
        "abstract": paper.description,
        "id_scopus": paper.eid,
        "type": paper.subtypeDescription,
        "keywords_scopus": paper.authkeywords,
    }
    for paper in scopus_search.results
]
pd.DataFrame(data).assign(
    "relevace_oreder",
)

In [ ]:
[x.title for x in scopus_search.results]

In [ ]:
scopus_api_key = os.environ.get("SCOPUS_API_KEY")
assert scopus_api_key is not None
url = "https://api.elsevier.com/content/search/scopus"
params = {"query": 'title("navigational risk assessment")', "apiKey": scopus_api_key}
headers = {"Accept": "application/json"}

response = requests.get(url, headers=headers, params=params)

In [ ]:
dict_an = response.json()
dict_an

In [ ]:
dict_an["search-results"].keys()

In [ ]:
import pandas as pd

df_all = pd.DataFrame(dict_an["search-results"]["entry"])
df_all.iloc[0].T

In [ ]:
[x.authkeywords for x in scopus_search.results]

In [ ]:
fig = px.bar(
    l_test_openalex.groupby("topic_openalex")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("count")
    .reset_index(),
    y="topic_openalex",
    x="count",
)
# fig.update_xaxes(tickangle=90)
fig